# Cosmo-Edu Lab: Galaxy Rotation Curves & Dark Matter 🌌

By observing how fast gas and stars are rotating around the galactic center, we can measure the total gravitational pull. We will then compare this to the gravity generated by the visible matter (stars and gas). The difference will reveal the hidden Dark Matter halo surrounding the galaxy!

### 💡 The Mission:
We will fit the observed rotation curves using the **Burkert Profile** for Dark Matter.


In [ ]:
import os
import glob
import io
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, root_scalar
import plotly.express as px
import plotly.graph_objects as go


# Gravitational constant in units of (kpc * (km/s)^2) / M_sun
G_grav = 4.30091e-6

# Local critical density of the Universe
H0 = 73.0
H0_kpc = H0 / 1000.0
rho_crit_local = (3.0 * H0_kpc**2) / (8.0 * np.pi * G_grav)
r200_factor = np.sqrt((800.0 * np.pi * G_grav * rho_crit_local) / 3.0)

print("Libraries imported and constants defined! Ready for physics. 🚀")



Libraries imported and constants defined! Ready for physics.


## Step 1: The Physics of Dark Matter Halos

To model the Dark Matter, we will use the **Burkert Profile**. Unlike other models, the Burkert profile assumes that the dark matter density flattens out near the center of the galaxy (creating a "core") rather than a cuspy profile like NFW.

The mass of Dark Matter enclosed within a radius $r$ is given by:

$$ M_{Burkert}(<r) = 2 \pi \rho_s r_s^3 \left[ \frac{1}{2}\ln\left(1 + \left(\frac{r}{r_s}\right)^2\right) + \ln\left(1+\frac{r}{r_s}\right) - \arctan\left(\frac{r}{r_s}\right) \right] $$

Where:
* **$r_s$** is the scale radius (how spread out the halo is).
* **$\rho_s$** is the central dark matter density.

In [ ]:
def M_burkert_enclosed(r, rho_s, r_s):
    """Calculates the Dark Matter mass enclosed within radius r using the Burkert profile."""
    x = r / r_s
    return 2.0 * np.pi * rho_s * (r_s**3) * (0.5 * np.log(1.0 + x**2) + np.log(1.0 + x) - np.arctan(x))

def observed_rho_from_RC(r, Vobs, Vgas, Vdisk, Vbul, y_disk=0.5, y_bul=0.7):
    """Calculates the observed density profile from the rotation curve."""
    # Splitted Mass-to-Light Ratio: 0.5 for Disk, 0.7 for Bulge
    V_bar_sq = Vgas * np.abs(Vgas) + y_disk * (Vdisk * np.abs(Vdisk)) + y_bul * (Vbul * np.abs(Vbul))
    V_DM_sq = np.maximum(Vobs**2 - V_bar_sq, 0)
    q = r**2 * (V_DM_sq / r)
    dq_dr = np.gradient(q, r)
    rho_DM = (1.0 / (4.0 * np.pi * G_grav * r**2)) * dq_dr
    return rho_DM

## Step 2: The Optimization Algorithm (Differential Evolution) 🧬

To find the correct parameters for each galaxy, deterministic algorithms (like SLSQP) often get trapped in "local minima" due to noisy data. Instead, we will use a global stochastic algorithm: **Differential Evolution (DE)**.

Think of it as a biological evolving ecosystem:
1. **Population:** We generate a population of candidate solutions (agents) randomly scattered across the parameter space.
2. **Mutation & Crossover:** Agents combine and mutate to explore new regions of the galaxy parameters.
3. **Selection:** Only the "fittest" agents (the ones that minimize the error between simulated and observed velocities) survive to the next generation!

**The Parameters & The MAP Framework:**
We are fitting 3 main parameters: $r_s$, $\rho_s$, and the stellar Mass-to-Light ratio **$\Upsilon_D$** (plus $\Upsilon_B$ if a bulge is present). 
To keep the stellar mass physically realistic, we use a **Maximum A Posteriori (MAP)** approach. We apply a Gaussian "penalty" to our Cost Function (Chi-Square) if the algorithm tries to push $\Upsilon$ too far from the theoretically expected values (0.5 for the disk, 0.7 for the bulge).

$$ \text{Cost} = \chi^2_{\text{kin}} + \left( \frac{\log_{10}\Upsilon_D - \log_{10}0.5}{0.1} \right)^2 $$

In [ ]:
def fit_galaxy_parameters(r, Vobs, errV, Vgas, Vdisk, Vbul):
    """
    Fits the dark matter profile using 3-Parameter Differential Evolution (DE 3P)
    evaluated within a Maximum A Posteriori (MAP) framework.
    """
    has_bulge = np.any(Vbul > 0)
    num_params = 4 if has_bulge else 3
    dof = max(1, len(r) - num_params)
    
  
    verr_safe = np.where(errV == 0, 0.1, errV) 

    # Objective function  (Cost = Chi2 + Lognormal Prior Penalty)
    def objective(params):
        log_rs, log_rhos, log_yd = params[0], params[1], params[2]
        rs, rho_s, yd = 10**log_rs, 10**log_rhos, 10**log_yd
        
        if has_bulge:
            log_yb = params[3]
            yb = 10**log_yb
        else:
            yb = 0.0
        
        # Barionic Velocity Contribution
        V_bar_sq_opt = Vgas * np.abs(Vgas) + yd * (Vdisk * np.abs(Vdisk)) + yb * (Vbul * np.abs(Vbul))
        
        # Dark Matter Velocity (Burkert)
        x_opt = r / rs
        M_dm_opt = 2.0 * np.pi * rho_s * (rs**3) * (0.5 * np.log(1.0 + x_opt**2) + np.log(1.0 + x_opt) - np.arctan(x_opt))
        V_tot_opt = np.sqrt(np.maximum(V_bar_sq_opt + (G_grav * M_dm_opt) / r, 0))
        
        # Chi-squared (Goodness of Fit)
        chi2_kin = np.sum(((Vobs - V_tot_opt) / verr_safe)**2)
        
        # Penalty MAP (Log-normal Prior on Upsilon)
        penalty = ((log_yd - np.log10(0.5)) / 0.1)**2
        if has_bulge:
            penalty += ((log_yb - np.log10(0.7)) / 0.1)**2
            
        return chi2_kin + penalty

    # Parameters space ( Log10) 
    if has_bulge:
        limiti = [
            (np.log10(0.01), np.log10(7000.0)),     # r_s
            (np.log10(3.2e4), np.log10(1.6e12)),    # rho_s
            (np.log10(0.1), np.log10(2.0)),         # Upsilon Disk
            (np.log10(0.1), np.log10(2.0))          # Upsilon Bulge
        ]
    else:
        limiti = [
            (np.log10(0.01), np.log10(7000.0)),
            (np.log10(3.2e4), np.log10(1.6e12)),
            (np.log10(0.1), np.log10(2.0))
        ]

    # DIFFERENTIAL EVOLUTION 
    res = differential_evolution(
            objective, 
            bounds=limiti, 
            strategy='rand1bin',   
            popsize=30,            
            tol=1e-8,            
            mutation=(0.5, 1.5),   
            recombination=0.5,     
            polish=True,           
            updating='deferred'
        )


    log_rs_fin, log_rhos_fin, log_yd_fin = res.x[0], res.x[1], res.x[2]
    r_s_fin, rho_s_fin, yd_fin = 10**log_rs_fin, 10**log_rhos_fin, 10**log_yd_fin
    yb_fin = 10**res.x[3] if has_bulge else 0.0

  
    V_bar_sq_fin = Vgas * np.abs(Vgas) + yd_fin * (Vdisk * np.abs(Vdisk)) + yb_fin * (Vbul * np.abs(Vbul))
    x_opt = r / r_s_fin
    M_dm_opt = 2.0 * np.pi * rho_s_fin * (r_s_fin**3) * (0.5 * np.log(1.0 + x_opt**2) + np.log(1.0 + x_opt) - np.arctan(x_opt))
    V_tot_fin = np.sqrt(np.maximum(V_bar_sq_fin + (G_grav * M_dm_opt) / r, 0))
    final_chi2 = np.sum(((Vobs - V_tot_fin) / verr_safe)**2) / dof

    # Cosmological parameters (V200, C200, M200)
    def find_c200(c):
        term = 0.5 * np.log(1.0 + c**2) + np.log(1.0 + c) - np.arctan(c)
        rho_s_teorica = (400.0 / 3.0) * rho_crit_local * (c**3) / term
        return rho_s_teorica - rho_s_fin

    try:
        res_c = root_scalar(find_c200, bracket=[1e-4, 1e5], method='brentq')
        c200_calc = res_c.root
        r200_calc = c200_calc * r_s_fin
        M200_fin = (800.0 / 3.0) * np.pi * rho_crit_local * (r200_calc**3)
        V200_fin = r200_calc * r200_factor
    except ValueError:
        c200_calc, V200_fin, M200_fin = 0.0, 0.0, 0.0 
    
    return final_chi2, rho_s_fin, r_s_fin, yd_fin, yb_fin, M200_fin, V200_fin, c200_calc

## Step 3: Running the Analysis Pipeline

It's time to process our data! Make sure you have uploaded the `Table.txt` file and the dataset `Rotmod_LTG` folder into your Colab environment.

The pipeline will:
1. Cross-reference the galaxies to remove non-ideal candidates (e.g., disk-halo degeneracy galaxies or those with poor data quality).
2. Fit the rotation curve for every single valid galaxy.
3. Plot the final rotation curve and output the results.

In [ ]:

input_folder = "/content/drive/MyDrive/Cosmo-Edu-Lab/Rotmod_LTG"
mrt_file = "Table.txt"

q_dict = {}
if os.path.exists(mrt_file):
    with open(mrt_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            sline = line.strip()
            if not sline or sline.startswith('-') or sline.startswith('=') or "Byte" in sline or "Note" in sline: continue
            parts = sline.split()
            if len(parts) >= 18 and parts[0].upper() != "GALAXY":
                name = parts[0].strip().upper()
                q_str = parts[17].strip()
                if q_str.isdigit():
                    q_dict[name] = {"Q": int(q_str)}

files = sorted(glob.glob(os.path.join(input_folder, "*.dat")))


results_data = []

print("="*115)
print(f"{'Galaxy':<15} | {'Q':<3} | {'Chi2 Min':<8} | {'rho_s':<9} | {'r_s':<6} | {'Ups D/B':<7} | {'M_tot_edge':<9} | {'% DM':>5}")
print("="*115)

for file in files:
    gal_name = os.path.splitext(os.path.basename(file))[0]
    gal_name_clean = gal_name.replace("_rotmod", "").strip().upper()
    gal_info = q_dict.get(gal_name_clean, {})
    
    # Galaxies filter for quality
    Q_val = gal_info.get("Q", "?")
    if Q_val in [2, 3]: continue
    
    try:
        with open(file, 'r') as f: lines = f.readlines()
        header_line, data_lines = None, []
        for line in lines:
            sline = line.strip()
            if not sline: continue
            if sline.startswith('#') and 'Rad' in sline and 'Vobs' in sline: 
                header_line = sline.replace('#', '').strip() + '\n'
            elif not sline.startswith('#'): 
                data_lines.append(line)
        
        if not header_line or not data_lines: continue
        df = pd.read_csv(io.StringIO(header_line + "".join(data_lines)), sep=r'\s+')
        if 'Rad' not in df.columns: continue
            
        r, Vobs = df['Rad'].values.astype(float), df['Vobs'].values.astype(float)
        errV = df['errV'].values.astype(float) if 'errV' in df.columns else np.ones_like(Vobs) * 5.0
        Vgas = df['Vgas'].values.astype(float) if 'Vgas' in df.columns else np.zeros_like(Vobs)
        Vdisk = df['Vdisk'].values.astype(float) if 'Vdisk' in df.columns else np.zeros_like(Vobs)
        Vbul = df['Vbul'].values.astype(float) if 'Vbul' in df.columns else np.zeros_like(Vobs)
        
        # Call Differential Evolution
        chi2_min, rho_s, r_s, y_disk, y_bul, m200, v200, c200 = fit_galaxy_parameters(r, Vobs, errV, Vgas, Vdisk, Vbul)
        
     
        V_bar_sq = Vgas * np.abs(Vgas) + y_disk * (Vdisk * np.abs(Vdisk)) + y_bul * (Vbul * np.abs(Vbul))
        V_bar = np.sqrt(np.maximum(V_bar_sq, 0))
        M_dm_grid = M_burkert_enclosed(r, rho_s, r_s)
        V_dm = np.sqrt(np.maximum((G_grav * M_dm_grid) / r, 0))
        V_sim = np.sqrt(np.maximum(V_bar_sq + (G_grav * M_dm_grid) / r, 0))
        
        M_bar_edge = (r[-1] * V_bar_sq[-1]) / G_grav
        M_tot_edge = M_bar_edge + M_dm_grid[-1]
        dm_frac_ratio = M_dm_grid[-1] / M_tot_edge if M_tot_edge > 0 else 0.0
        
       
        results_data.append({
            "Galaxy": gal_name_clean, 
            "M_tot_edge": M_tot_edge,
            "DM_Fraction_Edge": dm_frac_ratio
        })
        
      
        ups_str = f"{y_disk:.2f}/{y_bul:.2f}" if y_bul > 0 else f"{y_disk:.2f}"
        print(f"{gal_name_clean:<15} | {Q_val:<3} | {chi2_min:<8.2f} | {rho_s:<9.2e} | {r_s:<6.2f} | {ups_str:<7} | {M_tot_edge:<9.2e} | {dm_frac_ratio*100:>4.1f}%")
        
        
        rho_burkert_array = rho_s / ((1.0 + r / r_s) * (1.0 + (r / r_s)**2))
        
        #Plotting  rotational curves
        fig, ax1 = plt.subplots(figsize=(8, 4))
        
        ax1.errorbar(r, Vobs, yerr=errV, fmt='o', color='blue', ecolor='gray', label='Observations')
        ax1.plot(r, V_bar, '-', color='red', linewidth=2, label=f'Baryonic (MAP Fit)')
        ax1.plot(r, V_dm, '-', color='purple', linewidth=2, label='Dark Matter (Burkert)')
        ax1.plot(r, V_sim, '-', color='green', linewidth=2.5, label='Total Simulated')

        ax1.set_title(f'Rotation Curve: {gal_name_clean}', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Radius (kpc)', fontsize=12)
        ax1.set_ylabel('Velocity (km/s)', fontsize=12)
        ax1.set_ylim(0, max(max(Vobs), max(V_sim)) * 1.15)
        ax1.grid(True, linestyle='--', alpha=0.6)
        
        ax2 = ax1.twinx()
        ax2.plot(r, rho_burkert_array, '--', color='darkorange', linewidth=2, alpha=0.7, label='DM Density')
        ax2.set_ylabel(r'DM Density ($M_\odot / kpc^3$)', fontsize=12, color='darkorange')
        ax2.set_yscale('log')
        ax2.tick_params(axis='y', labelcolor='darkorange')
        
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='lower right', fontsize=9)
        
        plt.show() 
        
      

    except Exception as e:
        print(f"Errore su {gal_name_clean}: {e}")

## Step 4: Final Interactive Visualization

We calculated the Dark Matter fraction for every galaxy in our sample. Let's build an interactive scatter plot to explore the relationship between the Total Mass of the galaxy and its Dark Matter percentage.

**Instruction:** Hover your mouse over the data points to see the name of the specific galaxy!

In [ ]:


if len(results_data) > 0:
    # 1. Convert the list of results into a Pandas DataFrame
    df_results = pd.DataFrame(results_data)
    
    # 2. Calculate the percentage and add it as a new column
    df_results["DM_Perc"] = df_results["DM_Fraction_Edge"] * 100.0

    # 3. Create the interactive Plotly chart using our df_results
    fig_interactive = px.scatter(
        df_results,
        x="M_tot_edge",
        y="DM_Perc",
        hover_name="Galaxy",
        log_x=True,
        color_discrete_sequence=['royalblue'],
        labels={
            "M_tot_edge": "Total Mass (Solar Masses)",
            "DM_Perc": "Dark Matter Fraction (%)"
        },
        title="Dark Matter Fraction vs Total Mass (DE 3P Fit)"
    )

    fig_interactive.update_traces(marker=dict(size=12, line=dict(width=1, color='black'), opacity=0.8))
    fig_interactive.update_layout(title_font_size=18, xaxis_title_font_size=14, yaxis_title_font_size=14)

    
    fig_interactive.show()
else:
    print("No valid galaxies found to plot. Check your data upload or the pipeline execution.")